# 04 Build Patient Features

This notebook builds AI-ready patient feature tables from normalized vital signs. It keeps feature engineering separate from the normalized relational import layer.

In [1]:
import pandas as pd
from pathlib import Path

PROCESSED_DATA_DIR = Path('data/processed')
FEATURES_DATA_DIR = Path('data/features')

PATIENT_FILE = PROCESSED_DATA_DIR / 'patient.csv'
VITAL_SIGNS_FILE = PROCESSED_DATA_DIR / 'vital_signs.csv'

FEATURES_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
patient_df = pd.read_csv(PATIENT_FILE)
vital_signs_df = pd.read_csv(VITAL_SIGNS_FILE, parse_dates=['measured_at'])

print('Patient shape:', patient_df.shape)
print('Vital signs shape:', vital_signs_df.shape)

Patient shape: (333, 13)
Vital signs shape: (30390, 10)


In [3]:
latest_vitals = (
    vital_signs_df
    .sort_values(['patient_id', 'vital_type', 'measured_at'])
    .groupby(['patient_id', 'vital_type'], as_index=False)
    .tail(1)
)

latest_vitals.head()

,id,patient_id,vital_type,value,unit,measured_at,source_patient_id,source_encounter_id,observation_code,description
35,e0afd1fa-14f0-4f23-82a1-d90b1716f580,003d3430-65ba-41b9-a64b-49e1c9d2d7c8,BLOOD_PRESSURE_DIASTOLIC,92.0,mmHg,2025-12-11 08:35:40+00:00,23c94034-f050-04d5-c5a3-2d42bf3ccc7f,23c94034-f050-04d5-3e6f-b6b7263416ba,8462-4,Diastolic Blood Pressure
36,b5a57773-f9a9-48fd-82bd-56e278367eb2,003d3430-65ba-41b9-a64b-49e1c9d2d7c8,BLOOD_PRESSURE_SYSTOLIC,141.0,mmHg,2025-12-11 08:35:40+00:00,23c94034-f050-04d5-c5a3-2d42bf3ccc7f,23c94034-f050-04d5-3e6f-b6b7263416ba,8480-6,Systolic Blood Pressure
37,b4d3f1fb-88db-4080-94d5-004415921709,003d3430-65ba-41b9-a64b-49e1c9d2d7c8,HEART_RATE,87.0,beats/min,2025-12-11 08:35:40+00:00,23c94034-f050-04d5-c5a3-2d42bf3ccc7f,23c94034-f050-04d5-3e6f-b6b7263416ba,8867-4,Heart rate
38,d2277a23-60ec-44dd-b6ba-a285f75a258f,003d3430-65ba-41b9-a64b-49e1c9d2d7c8,HEIGHT,81.1,cm,2025-12-11 08:35:40+00:00,23c94034-f050-04d5-c5a3-2d42bf3ccc7f,23c94034-f050-04d5-3e6f-b6b7263416ba,8302-2,Body Height
39,53af40a5-9b7a-4dfb-8d6d-fdca27db3db9,003d3430-65ba-41b9-a64b-49e1c9d2d7c8,WEIGHT,10.7,kg,2025-12-11 08:35:40+00:00,23c94034-f050-04d5-c5a3-2d42bf3ccc7f,23c94034-f050-04d5-3e6f-b6b7263416ba,29463-7,Body Weight


In [4]:
latest_vitals_pivot = latest_vitals.pivot_table(
    index='patient_id',
    columns='vital_type',
    values='value',
    aggfunc='first'
).reset_index()

latest_vitals_pivot.columns.name = None
latest_vitals_pivot.head()

,patient_id,BLOOD_PRESSURE_DIASTOLIC,BLOOD_PRESSURE_SYSTOLIC,BMI,CHOLESTEROL,GLUCOSE,HEART_RATE,HEIGHT,OXYGEN_SATURATION,TEMPERATURE,WEIGHT
0,003d3430-65ba-41b9-a64b-49e1c9d2d7c8,92.0,141.0,NaN,NaN,NaN,87.0,81.1,NaN,NaN,10.7
1,02073516-fec9-4fde-849a-9f58f704fc1d,72.0,89.0,32.8,198.1,66.2,97.0,180.8,NaN,37.1,107.1
2,021cb6e0-c80a-417d-825e-3436f62ddb1b,84.0,115.0,30.3,150.3,NaN,99.0,159.4,NaN,NaN,76.9
3,03b98b89-3642-4f66-8983-139d81e73c7e,81.0,107.0,30.3,120.3,93.2,63.0,163.2,81.5,41.2,80.8
4,0409c72d-167e-4571-a807-b55cf3f828ff,87.0,116.0,30.4,132.0,NaN,79.0,173.4,NaN,NaN,91.2


In [5]:
recent_30d = vital_signs_df.copy()
max_ts = recent_30d['measured_at'].max()
cutoff_ts = max_ts - pd.Timedelta(days=30)
recent_30d = recent_30d[recent_30d['measured_at'] >= cutoff_ts].copy()

mean_30d = recent_30d.pivot_table(
    index='patient_id',
    columns='vital_type',
    values='value',
    aggfunc='mean'
).reset_index()

mean_30d = mean_30d.add_prefix('avg_30d_')
mean_30d = mean_30d.rename(columns={'avg_30d_patient_id': 'patient_id'})
mean_30d.head()

vital_type,patient_id,avg_30d_BLOOD_PRESSURE_DIASTOLIC,avg_30d_BLOOD_PRESSURE_SYSTOLIC,avg_30d_BMI,avg_30d_CHOLESTEROL,avg_30d_GLUCOSE,avg_30d_HEART_RATE,avg_30d_HEIGHT,avg_30d_OXYGEN_SATURATION,avg_30d_TEMPERATURE,avg_30d_WEIGHT
0,0409c72d-167e-4571-a807-b55cf3f828ff,87.0,116.0,30.4,NaN,NaN,79.0,173.4,NaN,NaN,91.2
1,069811dc-48e5-4be6-a981-c9fed26d1842,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,37.7,NaN
2,146711ad-38eb-4bcf-ad90-1b953d230631,84.0,106.0,27.4,NaN,91.4,76.0,160.8,NaN,NaN,70.8
3,1c540a3f-64d6-4c97-a02d-11aa6471250b,87.0,122.0,17.1,NaN,NaN,70.0,156.3,NaN,NaN,41.7
4,2866ee0b-00c4-4366-a2e4-dee5f1d5ebaa,83.0,106.0,29.3,232.9,107.2,81.0,180.4,NaN,NaN,95.3


In [6]:
weight_rows = vital_signs_df[vital_signs_df['vital_type'] == 'WEIGHT'].copy()
weight_rows = weight_rows.sort_values(['patient_id', 'measured_at'])

weight_trend = weight_rows.groupby('patient_id').agg(
    first_weight=('value', 'first'),
    latest_weight=('value', 'last'),
    first_weight_time=('measured_at', 'first'),
    latest_weight_time=('measured_at', 'last')
).reset_index()

weight_trend['weight_change'] = weight_trend['latest_weight'] - weight_trend['first_weight']
weight_trend.head()

,patient_id,first_weight,latest_weight,first_weight_time,latest_weight_time,weight_change
0,003d3430-65ba-41b9-a64b-49e1c9d2d7c8,3.7,10.7,2024-07-04 08:35:40+00:00,2025-12-11 08:35:40+00:00,7.0
1,02073516-fec9-4fde-849a-9f58f704fc1d,107.1,107.1,2013-12-11 21:23:58+00:00,2023-02-01 21:23:58+00:00,0.0
2,021cb6e0-c80a-417d-825e-3436f62ddb1b,76.9,76.9,2019-01-22 18:08:06+00:00,2025-01-28 18:08:06+00:00,0.0
3,03b98b89-3642-4f66-8983-139d81e73c7e,72.3,80.8,2016-07-12 23:09:42+00:00,2025-09-02 23:09:42+00:00,8.5
4,0409c72d-167e-4571-a807-b55cf3f828ff,93.1,91.2,2017-03-13 02:13:20+00:00,2026-03-23 02:13:20+00:00,-1.9


In [7]:
patient_features_df = patient_df[['id', 'patient_number', 'birth_date', 'gender']].copy()
patient_features_df = patient_features_df.rename(columns={'id': 'patient_id'})

patient_features_df = patient_features_df.merge(latest_vitals_pivot, on='patient_id', how='left')
patient_features_df = patient_features_df.merge(mean_30d, on='patient_id', how='left')
patient_features_df = patient_features_df.merge(weight_trend[['patient_id', 'weight_change', 'latest_weight_time']], on='patient_id', how='left')

patient_features_df.head()

,patient_id,patient_number,birth_date,gender,BLOOD_PRESSURE_DIASTOLIC,BLOOD_PRESSURE_SYSTOLIC,BMI,CHOLESTEROL,GLUCOSE,HEART_RATE,HEIGHT,OXYGEN_SATURATION,TEMPERATURE,WEIGHT,avg_30d_BLOOD_PRESSURE_DIASTOLIC,avg_30d_BLOOD_PRESSURE_SYSTOLIC,avg_30d_BMI,avg_30d_CHOLESTEROL,avg_30d_GLUCOSE,avg_30d_HEART_RATE,avg_30d_HEIGHT,avg_30d_OXYGEN_SATURATION,avg_30d_TEMPERATURE,avg_30d_WEIGHT,weight_change,latest_weight_time
0,535e0105-c571-4f45-be53-0ae18f605b0a,P00000001,2019-09-10,MALE,78.0,113.0,15.8,NaN,NaN,61.0,111.2,NaN,37.8,19.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.4,2025-08-26 08:06:56+00:00
1,77ca1acf-5d58-4574-8ba4-9957c932bc7a,P00000002,2009-08-17,MALE,88.0,128.0,18.9,NaN,NaN,87.0,170.7,NaN,37.5,55.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33.8,2025-09-29 02:57:01+00:00
2,a4a62cb0-c315-45a3-9d1e-52c88986cf8d,P00000003,1986-06-01,FEMALE,77.0,132.0,27.6,135.3,NaN,90.0,158.3,NaN,37.2,69.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,2023-08-19 22:33:58+00:00
3,f7a86fe6-7ee9-4698-b101-47f9afdd32e4,P00000004,1981-01-17,FEMALE,81.0,97.0,28.2,192.0,82.9,68.0,164.8,NaN,NaN,76.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-4.7,2025-02-01 19:16:29+00:00
4,d337e89d-2645-495d-82a3-d971292fd4c4,P00000005,1996-01-06,FEMALE,83.0,123.0,23.1,NaN,NaN,93.0,153.2,NaN,37.4,54.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.2,2025-03-22 05:51:11+00:00


In [8]:
feature_output_file = FEATURES_DATA_DIR / 'patient_features.csv'
patient_features_df.to_csv(feature_output_file, index=False)

print('Exported:', feature_output_file)

Exported: data\features\patient_features.csv


In [9]:
display(patient_features_df.head(20))

,patient_id,patient_number,birth_date,gender,BLOOD_PRESSURE_DIASTOLIC,BLOOD_PRESSURE_SYSTOLIC,BMI,CHOLESTEROL,GLUCOSE,HEART_RATE,HEIGHT,OXYGEN_SATURATION,TEMPERATURE,WEIGHT,avg_30d_BLOOD_PRESSURE_DIASTOLIC,avg_30d_BLOOD_PRESSURE_SYSTOLIC,avg_30d_BMI,avg_30d_CHOLESTEROL,avg_30d_GLUCOSE,avg_30d_HEART_RATE,avg_30d_HEIGHT,avg_30d_OXYGEN_SATURATION,avg_30d_TEMPERATURE,avg_30d_WEIGHT,weight_change,latest_weight_time
0,535e0105-c571-4f45-be53-0ae18f605b0a,P00000001,2019-09-10,MALE,78.0,113.0,15.8,NaN,NaN,61.0,111.2,NaN,37.8,19.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16.4,2025-08-26 08:06:56+00:00
1,77ca1acf-5d58-4574-8ba4-9957c932bc7a,P00000002,2009-08-17,MALE,88.0,128.0,18.9,NaN,NaN,87.0,170.7,NaN,37.5,55.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33.8,2025-09-29 02:57:01+00:00
2,a4a62cb0-c315-45a3-9d1e-52c88986cf8d,P00000003,1986-06-01,FEMALE,77.0,132.0,27.6,135.3,NaN,90.0,158.3,NaN,37.2,69.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,2023-08-19 22:33:58+00:00
3,f7a86fe6-7ee9-4698-b101-47f9afdd32e4,P00000004,1981-01-17,FEMALE,81.0,97.0,28.2,192.0,82.9,68.0,164.8,NaN,NaN,76.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-4.7,2025-02-01 19:16:29+00:00
4,d337e89d-2645-495d-82a3-d971292fd4c4,P00000005,1996-01-06,FEMALE,83.0,123.0,23.1,NaN,NaN,93.0,153.2,NaN,37.4,54.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.2,2025-03-22 05:51:11+00:00
5,30898761-15a4-47cd-9c61-fe9b6ff4d8a1,P00000006,2017-10-06,MALE,91.0,120.0,15.8,NaN,NaN,79.0,128.5,NaN,37.6,26.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.2,2025-10-03 01:33:26+00:00
6,8285a3e3-7a04-4413-8278-422379448f47,P00000007,2007-04-03,FEMALE,92.0,115.0,22.6,NaN,NaN,67.0,163.1,NaN,NaN,60.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,22.2,2025-05-27 03:19:07+00:00
7,6e4321e9-9f8a-4bd8-a9db-eeebead4c5df,P00000008,1975-05-11,FEMALE,76.0,120.0,30.1,252.2,NaN,97.0,152.6,95.0,37.7,70.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.2,2025-05-11 16:22:47+00:00
8,9d64ca04-3821-43fb-98d0-fa3f1672d7d4,P00000009,2023-04-03,FEMALE,81.0,129.0,16.9,NaN,NaN,71.0,96.4,NaN,NaN,15.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.4,2026-03-09 21:56:11+00:00
9,dc310d3e-46d0-411c-934e-ee99d1d91361,P00000010,2010-07-12,MALE,93.0,117.0,16.6,NaN,NaN,96.0,175.2,NaN,NaN,50.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,29.0,2025-08-18 05:59:38+00:00
